# Kind-LM Mechanistic Probe v2

## Environment Setup

In [ ]:
import subprocess, sys

packages = [
 "numpy<2.0.0", # pinned: C API binary incompatibility guard (Expected 96, got 88)
 "transformer_lens",
 "einops",
 "plotly",
 "jaxtyping",
 "transformers",
 "pandas",
 "kaleido==0.2.1", # pinned per Plotly compatibility warning
 "scipy", # entropy, FDR correction
 "tuned-lens", # v2: tuned lens cross-validation for Analysis 4
]

for pkg in packages:
 subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

print(" All packages installed.")


## Imports & Global Config


In [ ]:
import os
import gc
import json
import shutil
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import plotly.express as px
import plotly.graph_objects as go
from scipy.stats import entropy as scipy_entropy
from transformers import GPT2LMHeadModel, AutoTokenizer, GPT2Config
from transformer_lens import HookedTransformer

# Directories
os.makedirs("results_v2/figures", exist_ok=True)
os.makedirs("results_v2/data", exist_ok=True)

# Device
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f" Running on: {DEVICE}")
if DEVICE == "cuda":
 print(f" GPU: {torch.cuda.get_device_name(0)}")
 print(f" VRAM available: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Checkpoint registry (verified against live HuggingFace)
REPO_BASE = "llm-slice/blm-gpt2s-90M-s42"

# Dense coverage around the hypothesised threshold (20M–50M words)
PRETRAINING_REVISIONS = [
 "chck_1M",
 "chck_5M",
 "chck_10M",
 "chck_20M",
 "chck_30M",
 "chck_40M",
 "chck_50M",
 "chck_60M",
 "chck_100M",
 "chck_200M",
 "chck_1000M",
]

# PPO interaction checkpoints - separate repos, canonical seed=42
PPO_REPOS = {
 "50M+PPO": "llm-slice/blm-gpt2s-90M-s42_chck_50M_ppo-1000K-seed42",
 "200M+PPO": "llm-slice/blm-gpt2s-90M-s42_chck_200M_ppo-1000K-seed42",
}

# Checkpoints to run in the logit-lens and coreference analyses
# (subset to save GPU time - full sweep would take too long)
MECH_CHECKPOINTS = ["chck_10M", "chck_20M", "chck_50M", "chck_200M", "chck_1000M"]

# Expected custom vocabulary size - asserted on every load
EXPECTED_VOCAB = 16_000

# Story prompt (domain-matched to BabyLM training data)
STORY_PROMPT = "Once upon a time in a faraway land, there lived a"

# v2 parameters
N_INDUCTION_SAMPLES = 25 # number of random repeated sequences for induction score
INDUCTION_SEQ_LEN = 30 # half-length of repeated sequence; full sequence = 60 tokens
EPS_THRESHOLD = 0.5 # KL threshold for "model has committed"

print(" Config loaded.")


## Robust Model Loading Utility

**Critical vocab alignment note:** `HookedTransformer.from_pretrained("gpt2")` would load
embedding matrices for vocab=50,257. BabyLM models use vocab=16,000.
We load the native HuggingFace model first and pipe it through `hf_model=` so
TransformerLens inherits the correct weight shapes. An assert guards every call.

In [ ]:
def load_model(
    repo_id: str, revision: str = "main", random_init: bool = False
) -> HookedTransformer:
    print(f" Loading {'RANDOM INIT' if random_init else revision} from {repo_id}")

    config_revision = revision if not random_init else "chck_1M"
    tokenizer = AutoTokenizer.from_pretrained(repo_id, revision=config_revision)

    if random_init:
        config = GPT2Config.from_pretrained(repo_id, revision=config_revision)
        hf_model = GPT2LMHeadModel(config)
    else:
        hf_model = GPT2LMHeadModel.from_pretrained(repo_id, revision=revision)

    tl_model = HookedTransformer.from_pretrained(
        "gpt2",
        hf_model=hf_model,
        tokenizer=tokenizer,
        device=DEVICE,
        n_ctx=hf_model.config.n_positions,
        d_vocab=hf_model.config.vocab_size,
    )

    assert tl_model.cfg.d_vocab == EXPECTED_VOCAB, (
        f" Vocab mismatch! Got {tl_model.cfg.d_vocab}, expected {EXPECTED_VOCAB}. "
        f"Check revision '{revision}' - tokenizer must come from a chck_* branch."
    )

    tl_model.eval()
    return tl_model


def release(model: HookedTransformer):
    model.reset_hooks(clear_contexts=True)
    del model
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.synchronize()


def vram_gb() -> float:
    if DEVICE == "cuda":
        return torch.cuda.memory_allocated() / 1e9
    return 0.0


def validate_single_token(model: HookedTransformer, word: str) -> int:
    tokens = model.to_tokens(word, prepend_bos=False)[0]
    if len(tokens) > 1:
        readable = model.to_str_tokens(tokens)
        raise ValueError(f"'{word}' {len(tokens)} tokens: {readable}")
    return tokens[0].item()


print(" Model loading utilities defined.")

## Analysis 1 - Induction Head Mapping

 used a single fixed 5-token repeated sequence and reported
`max_score` across all heads - easily dominated by one noisy head.

Score is now the **mean** across N=25 independent random repeated sequences
of length 2×30 tokens, averaged over all positions in the repeated half. This gives a
robust, replication-stable induction head measurement consistent with Olsson et al. (2022).

**Hypothesis:** The 20M–50M threshold coincides with the emergence of induction heads
(S-curve in the max induction score). The 20M spike followed by a 30M–50M dip is
predicted to reflect circuit competition during representational reorganisation.

In [ ]:
def get_induction_scores_robust(
    model: HookedTransformer,
    n_samples: int = N_INDUCTION_SAMPLES,
    seq_len: int = INDUCTION_SEQ_LEN,
) -> torch.Tensor:
    model.reset_hooks()
    n_layers = model.cfg.n_layers
    n_heads = model.cfg.n_heads
    accumulated = torch.zeros(n_layers, n_heads)

    for _ in range(n_samples):
        base = torch.randint(100, 15_000, (1, seq_len))
        tokens = torch.cat([base, base], dim=1).to(DEVICE)  # [1, 2*seq_len]

        _, cache = model.run_with_cache(
            tokens,
            names_filter=lambda name: "pattern" in name,
        )

        for layer in range(n_layers):
            pattern = cache["pattern", layer][0]  # [n_heads, 2*seq_len, 2*seq_len]
            for i in range(seq_len - 1):
                accumulated[layer] += pattern[:, seq_len + i, i + 1].cpu()

    scores = accumulated / (n_samples * (seq_len - 1))
    model.reset_hooks()
    return scores


print(f" Analysis 1: Induction Head Sweep - {N_INDUCTION_SAMPLES} sequences each")
induction_rows = []

print(" [0/12] Random init baseline ...")
m = load_model(REPO_BASE, random_init=True)
scores = get_induction_scores_robust(m)
induction_rows.append(
    {
        "checkpoint": "0M_random",
        "max_score": scores.max().item(),
        "mean_score": scores.mean().item(),
        "top_head_layer": scores.argmax().item() // m.cfg.n_heads,
        "top_head_idx": scores.argmax().item() % m.cfg.n_heads,
        "scores_json": scores.tolist(),
    }
)
release(m)
print(f" VRAM after release: {vram_gb():.2f} GB")

for i, rev in enumerate(PRETRAINING_REVISIONS, 1):
    print(f" [{i}/{len(PRETRAINING_REVISIONS)}] {rev} ...")
    m = load_model(REPO_BASE, revision=rev)
    scores = get_induction_scores_robust(m)
    induction_rows.append(
        {
            "checkpoint": rev.replace("chck_", ""),
            "max_score": scores.max().item(),
            "mean_score": scores.mean().item(),
            "top_head_layer": int(scores.argmax().item() // m.cfg.n_heads),
            "top_head_idx": int(scores.argmax().item() % m.cfg.n_heads),
            "scores_json": scores.tolist(),
        }
    )
    release(m)
    print(f" VRAM after release: {vram_gb():.2f} GB")

df_induction = pd.DataFrame(induction_rows)
df_induction.to_csv("results_v2/data/analysis_1_induction_scores.csv", index=False)
print(" Analysis 1 complete - data saved.")

fig1a = px.line(
    df_induction,
    x="checkpoint",
    y="max_score",
    markers=True,
    title="Analysis 1: Peak Induction Score - Averaged over 25 Random Sequences",
    labels={
        "checkpoint": "Pretraining Corpus Size (words)",
        "max_score": "Mean-Robust Max Induction Score",
    },
    template="plotly_white",
)
fig1a.add_vrect(
    x0="20M",
    x1="50M",
    fillcolor="orange",
    opacity=0.15,
    annotation_text="Hypothesised threshold",
    annotation_position="top left",
)
fig1a.write_html("results_v2/figures/analysis_1_induction_trajectory.html")
fig1a.show()

fig1b = px.line(
    df_induction,
    x="checkpoint",
    y=["max_score", "mean_score"],
    markers=True,
    title="Analysis 1: Max vs Mean Induction Score",
    labels={
        "checkpoint": "Pretraining Size (words)",
        "value": "Induction Score",
        "variable": "Metric",
    },
    template="plotly_white",
)
fig1b.add_vrect(x0="20M", x1="50M", fillcolor="orange", opacity=0.15)
fig1b.write_html("results_v2/figures/analysis_1_max_vs_mean.html")
fig1b.show()

## Analysis 2 - PPO Causal Isolation

 reported only `Δmax` (a single unsigned number), making it
impossible to know *which* heads were affected or *whether* PPO raised or lowered scores.

Full signed delta `Δ = score_PPO − score_base` is rendered as a 2D
Layer × Head heatmap using a diverging RdBu scale. Red = PPO raised the score
(reinforced induction/copy behaviour); Blue = PPO suppressed it.

**Prediction:** PPO trained on preference/story feedback should *suppress* pure copy
heads (blue) because reward models penalise repetition. If we see red, PPO is
reinforcing the induction circuit - a different hypothesis.

In [ ]:
print(" Analysis 2: PPO Causal Isolation - Signed Per-Head Delta Heatmaps")

ppo_records = []

for label, (base_rev, ppo_repo) in {
    "50M": ("chck_50M", PPO_REPOS["50M+PPO"]),
    "200M": ("chck_200M", PPO_REPOS["200M+PPO"]),
}.items():
    print(f" Loading base {base_rev} ...")
    m_base = load_model(REPO_BASE, revision=base_rev)
    scores_base = get_induction_scores_robust(m_base)
    release(m_base)

    print(f" Loading PPO model {ppo_repo} ...")
    m_ppo = load_model(ppo_repo, revision="main")
    scores_ppo = get_induction_scores_robust(m_ppo)
    release(m_ppo)

    delta = scores_ppo - scores_base
    torch.save(delta, f"results_v2/data/analysis_2_delta_{label}vsPPO.pt")
    torch.save(scores_base, f"results_v2/data/analysis_2_base_{label}.pt")
    torch.save(scores_ppo, f"results_v2/data/analysis_2_ppo_{label}.pt")

    n_layers, n_heads = delta.shape
    for layer in range(n_layers):
        for head in range(n_heads):
            ppo_records.append({
                "base": label,
                "layer": layer,
                "head": head,
                "score_base": scores_base[layer, head].item(),
                "score_ppo": scores_ppo[layer, head].item(),
                "delta_induction": delta[layer, head].item(),
            })

    print(f" max={delta.abs().max():.4f} "
          f"pos={delta[delta>0].sum():.4f} "
          f"neg={delta[delta<0].sum():.4f}")
    print(f" VRAM after release: {vram_gb():.2f} GB")

df_ppo = pd.DataFrame(ppo_records)
df_ppo.to_csv("results_v2/data/analysis_2_ppo_delta.csv", index=False)
print(" Analysis 2 complete - data saved.")

for base_label in ["50M", "200M"]:
    sub = df_ppo[df_ppo["base"] == base_label]
    pivot = sub.pivot(index="layer", columns="head", values="delta_induction")

    fig2 = px.imshow(
        pivot,
        color_continuous_scale="RdBu",
        color_continuous_midpoint=0,
        title=f"Analysis 2: Signed Delta Induction Score (PPO - Base) @ {base_label}",
        labels={"x": "Attention Head", "y": "Layer", "color": "Delta Induction Score"},
        template="plotly_white",
        text_auto=".2f",
    )
    fig2.update_xaxes(side="top")
    fig2.write_html(f"results_v2/figures/analysis_2_signed_delta_{base_label}.html")
    fig2.show()


## Analysis 3 - Entity Coreference Tracking

All probes used words like " wizard" that the
custom 16K BPE tokenizer splits into multiple sub-tokens, causing every probe to be
skipped silently. This rendered the entire analysis null.

New probe set uses only words verified to be single tokens in the
GPT-2 16K vocabulary. A pre-flight validation assert now hard-fails if any probe
word becomes multi-token, preventing silent null results.

**Added:** Extended to MECH_CHECKPOINTS (5 points) instead of just 3, giving a
cleaner view of when coreference tracking first exceeds chance.

In [ ]:
COREFERENCE_PROBES_V2 = [
    {"prompt": "The boy gave the ball to the girl . The ball belonged to the",
     "correct": " boy", "foil": " girl"},
    {"prompt": "The king gave the ring to the queen . The ring belonged to the",
     "correct": " king", "foil": " queen"},
    {"prompt": "The dog followed the cat into the wood . The cat hid from the",
     "correct": " dog", "foil": " cat"},
    {"prompt": "The man and the woman walked down the road . The woman waved at the",
     "correct": " man", "foil": " woman"},
    {"prompt": "John went to the store . John bought",
     "correct": " milk", "foil": " shoes"},
    {"prompt": "Mary saw the dog . Mary",
     "correct": " pet", "foil": " hid"},
    {"prompt": "Tom read the book . Tom liked",
     "correct": " it", "foil": " nothing"},
    {"prompt": "Anna cooked dinner . Anna served",
     "correct": " it", "foil": " them"},
    {"prompt": "Sam drove fast . Sam reached",
     "correct": " home", "foil": " work"},
]


def preflight_probe_validation(model: HookedTransformer, probes: list[dict]) -> list[dict]:
    valid = []
    for p in probes:
        try:
            validate_single_token(model, p["correct"])
            validate_single_token(model, p["foil"])
            valid.append(p)
        except ValueError as e:
            print(f" Preflight skip: {e}")
    print(f" {len(valid)}/{len(probes)} probes passed preflight validation")
    return valid


print(" Analysis 3: Entity Coreference Tracking - Verified Single-Token Probes")
coref_rows = []

for rev in MECH_CHECKPOINTS:
    print(f" {rev} ...")
    m = load_model(REPO_BASE, revision=rev)
    valid_probes = preflight_probe_validation(m, COREFERENCE_PROBES_V2)

    for probe in valid_probes:
        correct_id = validate_single_token(m, probe["correct"])
        foil_id = validate_single_token(m, probe["foil"])
        tokens = m.to_tokens(probe["prompt"])
        with torch.no_grad():
            logits = m(tokens)
        logit_diff = (logits[0, -1, correct_id] - logits[0, -1, foil_id]).item()

        coref_rows.append({
            "checkpoint": rev.replace("chck_", ""),
            "probe_correct": probe["correct"].strip(),
            "probe_foil": probe["foil"].strip(),
            "prompt_snippet": probe["prompt"][-40:],
            "logit_diff": logit_diff,
            "above_chance": logit_diff > 0,
        })

    release(m)
    print(f" VRAM after release: {vram_gb():.2f} GB")

df_coref = pd.DataFrame(coref_rows)
df_coref.to_csv("results_v2/data/analysis_3_coreference_logits.csv", index=False)
print(" Analysis 3 complete - data saved.")

coref_summary = df_coref.groupby("checkpoint")["logit_diff"].agg(["mean", "std", "count"])
print("\nCoreference logit diff (mean +/- std) per checkpoint:")
print(coref_summary.to_string())

fig3 = px.bar(
    df_coref,
    x="checkpoint",
    y="logit_diff",
    color="probe_correct",
    barmode="group",
    title="Analysis 3: Entity Coreference Logit Difference",
    labels={"checkpoint": "Pretraining Size (words)", "logit_diff": "Logit Diff (correct - foil)", "probe_correct": "Correct completion"},
    template="plotly_white",
)
fig3.add_hline(y=0, line_dash="dash", line_color="grey", annotation_text="Chance level")
fig3.write_html("results_v2/figures/analysis_3_coreference.html")
fig3.show()

fig3b = px.line(
    coref_summary.reset_index(),
    x="checkpoint",
    y="mean",
    markers=True,
    error_y="std",
    title="Analysis 3: Mean Coreference Logit Difference (+/-1 SD across probes)",
    labels={"checkpoint": "Pretraining Size (words)", "mean": "Mean Logit Diff"},
    template="plotly_white",
)
fig3b.add_hline(y=0, line_dash="dash", line_color="grey")
fig3b.write_html("results_v2/figures/analysis_3_coref_mean_sd.html")
fig3b.show()


## Analysis 4 - Logit Lens KL Depth

Commit layer progression was 1 → 2 → 10 for 10M → 50M → 1000M words.

The basic logit lens has a known "representation mismatch" problem:
intermediate hidden states are in a different rotational basis than the final unembedding
matrix expects. A "commit layer = 1" at 10M may be an artifact, not a real signal.

After running the standard logit lens, we also check via the
`tuned-lens` library, which trains a per-layer affine translator to correct the mismatch.
If both agree, the result is reliable. If they diverge, the tuned lens result is more trustworthy.

We run on MECH_CHECKPOINTS (5 points) for richer resolution of the phase transition.

In [ ]:
EPS_THRESHOLD = 0.5
TUNED_LENS_CALIB_N = 50


def logit_lens_kl(model: HookedTransformer, prompt: str) -> list[float]:
    tokens = model.to_tokens(prompt)
    _, cache = model.run_with_cache(tokens, names_filter=lambda n: "resid_post" in n)
    with torch.no_grad():
        final_logits = model(tokens)[0, -1]
        final_probs = F.softmax(final_logits, dim=-1)

    kl_per_layer = []
    for layer in range(model.cfg.n_layers):
        resid = cache["resid_post", layer][0, -1]
        ln_out = model.ln_final(resid.unsqueeze(0)).squeeze(0)
        logits_l = ln_out @ model.W_U
        probs_l = F.softmax(logits_l, dim=-1)
        kl = F.kl_div(probs_l.log(), final_probs, reduction="sum").item()
        kl_per_layer.append(kl)

    return kl_per_layer


def try_tuned_lens_kl(model: HookedTransformer, prompt: str, calib_n: int = TUNED_LENS_CALIB_N):
    try:
        from tuned_lens.nn import TunedLens
        import random

        calib_tokens = torch.randint(100, EXPECTED_VOCAB, (calib_n, 32), device=DEVICE)
        tl = TunedLens(model).to(DEVICE)

        opt = torch.optim.Adam(tl.parameters(), lr=1e-3)
        for step in range(200):
            idx = random.randint(0, calib_n - 1)
            toks = calib_tokens[idx:idx+1]
            _, cache_calib = model.run_with_cache(toks, names_filter=lambda n: "resid_post" in n)
            with torch.no_grad():
                target = F.softmax(model(toks)[0, -1], dim=-1)
            loss = 0.0
            for L in range(model.cfg.n_layers):
                resid = cache_calib["resid_post", L][0, -1]
                pred = F.softmax(tl(resid, L), dim=-1)
                loss += F.kl_div(pred.log(), target, reduction="sum")
            opt.zero_grad()
            loss.backward()
            opt.step()

        tokens = model.to_tokens(prompt)
        _, cache_probe = model.run_with_cache(tokens, names_filter=lambda n: "resid_post" in n)
        with torch.no_grad():
            final_probs = F.softmax(model(tokens)[0, -1], dim=-1)

        kl_tuned = []
        with torch.no_grad():
            for L in range(model.cfg.n_layers):
                resid = cache_probe["resid_post", L][0, -1]
                probs = F.softmax(tl(resid, L), dim=-1)
                kl = F.kl_div(probs.log(), final_probs, reduction="sum").item()
                kl_tuned.append(kl)

        del tl
        gc.collect()
        return kl_tuned

    except Exception as e:
        print(f" Tuned lens skipped ({e.__class__.__name__}: {e})")
        return None


print(" Analysis 4: Logit Lens KL Depth + Tuned Lens Cross-Validation")
kl_rows = []

for rev in MECH_CHECKPOINTS:
    print(f" {rev} ...")
    m = load_model(REPO_BASE, revision=rev)

    klds_std = logit_lens_kl(m, STORY_PROMPT)
    klds_tuned = try_tuned_lens_kl(m, STORY_PROMPT)

    commit_std = next((i for i, v in enumerate(klds_std) if v < EPS_THRESHOLD), m.cfg.n_layers)
    commit_tuned = next((i for i, v in enumerate(klds_tuned) if v < EPS_THRESHOLD), m.cfg.n_layers) if klds_tuned else None

    release(m)
    print(f" Standard commit_layer={commit_std} | Tuned commit_layer={commit_tuned}")
    print(f" VRAM after release: {vram_gb():.2f} GB")

    for layer in range(len(klds_std)):
        kl_rows.append({
            "checkpoint": rev.replace("chck_", ""),
            "layer": layer,
            "kl_divergence": klds_std[layer],
            "kl_tuned": klds_tuned[layer] if klds_tuned else None,
            "commit_standard": commit_std,
            "commit_tuned": commit_tuned,
        })

df_kl = pd.DataFrame(kl_rows)
df_kl.to_csv("results_v2/data/analysis_4_logit_lens_kl.csv", index=False)
print(" Analysis 4 complete - data saved.")

fig4 = px.line(
    df_kl, x="layer", y="kl_divergence", color="checkpoint", markers=True,
    title="Analysis 4: Logit Lens KL-Divergence per Layer",
    labels={"layer": "Layer Index", "kl_divergence": "KL Divergence (vs final output)"},
    template="plotly_white",
)
fig4.add_hline(y=EPS_THRESHOLD, line_dash="dot", line_color="red",
               annotation_text=f"Commit threshold (e={EPS_THRESHOLD})")
fig4.write_html("results_v2/figures/analysis_4_logit_lens_standard.html")
fig4.show()

if df_kl["kl_tuned"].notna().any():
    fig4b = px.line(
        df_kl.dropna(subset=["kl_tuned"]),
        x="layer", y="kl_tuned", color="checkpoint", markers=True,
        title="Analysis 4: Tuned Lens KL-Divergence per Layer",
        labels={"layer": "Layer Index", "kl_tuned": "KL Divergence (tuned lens)"},
        template="plotly_white",
    )
    fig4b.add_hline(y=EPS_THRESHOLD, line_dash="dot", line_color="red")
    fig4b.write_html("results_v2/figures/analysis_4_logit_lens_tuned.html")
    fig4b.show()


## Analysis 5 - Direct Logit Attribution (DLA)

**What it is:** DLA decomposes the residual stream linearly - each attention head's output
vector is projected into logit space via the unembedding matrix `W_U`. The result is a
per-head "responsibility score" for the correct vs foil token at the final position.

**Why it matters:** If induction heads are causing better coreference, DLA provides
direct evidence: the responsible heads should show large, positive attribution for the
correct completion token. At 10M (pre-threshold), attribution should be diffuse and
noisy. At 50M+, it should concentrate on 1–2 specific heads.

**Output:** Per-checkpoint, per-head logit attribution heatmap (Layer × Head).
This is the "credit assignment map" for coreference resolution.

**Cost:** One full forward pass per probe per checkpoint - very fast.

In [ ]:
def compute_dla(model: HookedTransformer, probe: dict) -> torch.Tensor:
    correct_id = validate_single_token(model, probe["correct"])
    foil_id = validate_single_token(model, probe["foil"])
    tokens = model.to_tokens(probe["prompt"])

    _, cache = model.run_with_cache(
        tokens,
        names_filter=lambda n: "z" in n,
    )

    n_layers = model.cfg.n_layers
    n_heads = model.cfg.n_heads
    dla_mat = torch.zeros(n_layers, n_heads)

    W_U = model.W_U.detach()  # [d_model, vocab]

    for layer in range(n_layers):
        z = cache["z", layer][0, -1]  # [n_heads, d_head]
        W_O = model.W_O[layer]  # [n_heads, d_head, d_model]
        head_out = torch.einsum("hd,hdm->hm", z, W_O)  # [n_heads, d_model]

        logit_correct = (head_out @ W_U[:, correct_id])  # [n_heads]
        logit_foil = (head_out @ W_U[:, foil_id])  # [n_heads]
        dla_mat[layer] = (logit_correct - logit_foil).cpu()

    model.reset_hooks()
    return dla_mat


print(" Analysis 5: Direct Logit Attribution")
dla_rows = []
MECH_CHECKPOINTS_DLA = MECH_CHECKPOINTS

for rev in MECH_CHECKPOINTS_DLA:
    print(f" {rev} ...")
    m = load_model(REPO_BASE, revision=rev)
    valid_probes = preflight_probe_validation(m, COREFERENCE_PROBES_V2)

    if not valid_probes:
        print(f" No valid probes at {rev} - skipping.")
        release(m)
        continue

    dla_accum = torch.zeros(m.cfg.n_layers, m.cfg.n_heads)
    for probe in valid_probes:
        dla_accum += compute_dla(m, probe)
    dla_avg = dla_accum / len(valid_probes)

    torch.save(dla_avg, f"results_v2/data/analysis_5_dla_{rev}.pt")

    for layer in range(m.cfg.n_layers):
        for head in range(m.cfg.n_heads):
            dla_rows.append({
                "checkpoint": rev.replace("chck_", ""),
                "layer": layer,
                "head": head,
                "dla_score": dla_avg[layer, head].item(),
            })

    release(m)
    print(f" VRAM after release: {vram_gb():.2f} GB")

df_dla = pd.DataFrame(dla_rows)
df_dla.to_csv("results_v2/data/analysis_5_dla.csv", index=False)
print(" Analysis 5 complete - data saved.")

for ckpt in df_dla["checkpoint"].unique():
    sub = df_dla[df_dla["checkpoint"] == ckpt]
    pivot = sub.pivot(index="layer", columns="head", values="dla_score")

    fig5 = px.imshow(
        pivot,
        color_continuous_scale="RdBu",
        color_continuous_midpoint=0,
        title=f"Analysis 5: Direct Logit Attribution @ {ckpt} words",
        labels={"x": "Attention Head", "y": "Layer", "color": "DLA (correct - foil)"},
        template="plotly_white",
        text_auto=".2f",
    )
    fig5.write_html(f"results_v2/figures/analysis_5_dla_{ckpt}.html")
    fig5.show()


## Analysis 6 - Attention Entropy Sweep

**What it is:** For every attention head, compute the Shannon entropy of its
attention distribution at the final query position. High entropy = diffuse/broad
(attending equally to everything, no specialisation). Low entropy = sharp/focused
(attending to a few specific positions, a specialised circuit).

**Why it matters:** It is a zero-compute proxy for head specialisation. If a
bimodal entropy jump occurs at the 20M checkpoint - some heads becoming very sharp
while others remain diffuse - this is direct evidence of bifurcation into specialised
circuits at the threshold.

**Cost:** Only requires `cache["pattern", layer]` - already computed during induction
scoring. We run one additional pass here for the MECH_CHECKPOINTS subset.

In [ ]:
def compute_attention_entropy(model: HookedTransformer, prompt: str) -> torch.Tensor:
    tokens = model.to_tokens(prompt)
    _, cache = model.run_with_cache(tokens, names_filter=lambda n: "pattern" in n)

    n_layers = model.cfg.n_layers
    n_heads = model.cfg.n_heads
    ent_matrix = torch.zeros(n_layers, n_heads)

    for layer in range(n_layers):
        pattern = cache["pattern", layer][0]  # [n_heads, q_pos, k_pos]
        last_row = pattern[:, -1, :].cpu().numpy()  # [n_heads, k_pos]
        for head in range(n_heads):
            p = last_row[head]
            p = np.maximum(p, 1e-10)
            ent_matrix[layer, head] = float(scipy_entropy(p))

    model.reset_hooks()
    return ent_matrix


print(" Analysis 6: Attention Entropy Sweep")
entropy_rows = []

for rev in MECH_CHECKPOINTS:
    print(f" {rev} ...")
    m = load_model(REPO_BASE, revision=rev)
    ent = compute_attention_entropy(m, STORY_PROMPT)
    torch.save(ent, f"results_v2/data/analysis_6_entropy_{rev}.pt")

    for layer in range(m.cfg.n_layers):
        for head in range(m.cfg.n_heads):
            entropy_rows.append({
                "checkpoint": rev.replace("chck_", ""),
                "layer": layer,
                "head": head,
                "entropy": ent[layer, head].item(),
            })

    release(m)
    print(f" VRAM after release: {vram_gb():.2f} GB")

df_entropy = pd.DataFrame(entropy_rows)
df_entropy.to_csv("results_v2/data/analysis_6_entropy.csv", index=False)
print(" Analysis 6 complete - data saved.")

for ckpt in df_entropy["checkpoint"].unique():
    sub = df_entropy[df_entropy["checkpoint"] == ckpt]
    pivot = sub.pivot(index="layer", columns="head", values="entropy")
    fig6 = px.imshow(
        pivot,
        color_continuous_scale="Viridis_r",
        title=f"Analysis 6: Attention Entropy @ {ckpt} words",
        labels={"x": "Attention Head", "y": "Layer", "color": "Shannon Entropy"},
        template="plotly_white",
    )
    fig6.write_html(f"results_v2/figures/analysis_6_entropy_{ckpt}.html")
    fig6.show()

fig6b = px.box(
    df_entropy,
    x="checkpoint",
    y="entropy",
    title="Analysis 6: Distribution of Head Entropy Across Checkpoints",
    labels={"checkpoint": "Pretraining Size (words)", "entropy": "Shannon Entropy"},
    template="plotly_white",
    points="all",
)
fig6b.write_html("results_v2/figures/analysis_6_entropy_distribution.html")
fig6b.show()


## Analysis 7 - Activation Patching / Causal Tracing

**What it is:** The gold-standard mechanistic interpretability experiment.
We define a "clean" coreference prompt and a "corrupted" prompt (entity is swapped).
For each (layer, position) combination, we patch the clean residual stream
activations into the corrupted run and measure how much the logit difference recovers.

A high recovery at (layer L, position P) means: "this layer, at this token position,
is causally responsible for tracking the entity."

**Prediction:**
- 10M model: low, diffuse recovery (no causal localisation → no circuit)
- 50M model: concentrated recovery around the subject token position and
 a specific mid-network layer → circuit has formed
- 1000M model: even more concentrated recovery

If this prediction holds, it is direct experimental proof of the circuit emergence
hypothesis - not just correlation, but causal mechanism.

**Cost:** n_layers × n_positions forward passes per checkpoint.
~12 × 20 = 240 passes per checkpoint = ~4 min total on A100. Acceptable.

In [ ]:
PATCH_PROBE = {
    "clean": "The boy gave the ball to the girl . The ball belonged to the boy",
    "corrupted": "The boy gave the ball to the girl . The ball belonged to the girl",
    "correct_word": " boy",
    "foil_word": " girl",
}


def run_activation_patching(
    model: HookedTransformer,
    clean: str,
    corrupted: str,
    correct_w: str,
    foil_w: str,
) -> torch.Tensor:
    correct_id = validate_single_token(model, correct_w)
    foil_id = validate_single_token(model, foil_w)

    clean_toks = model.to_tokens(clean)
    corr_toks = model.to_tokens(corrupted)

    min_len = min(clean_toks.shape[1], corr_toks.shape[1])
    clean_toks = clean_toks[:, :min_len]
    corr_toks = corr_toks[:, :min_len]

    _, clean_cache = model.run_with_cache(
        clean_toks,
        names_filter=lambda n: "resid_post" in n,
    )

    with torch.no_grad():
        corr_logits = model(corr_toks)[0, -1]
        baseline_diff = (corr_logits[correct_id] - corr_logits[foil_id]).item()

    with torch.no_grad():
        clean_logits = model(clean_toks)[0, -1]
        clean_diff = (clean_logits[correct_id] - clean_logits[foil_id]).item()

    n_layers = model.cfg.n_layers
    n_pos = min_len
    patch_mat = torch.zeros(n_layers, n_pos)

    for layer in range(n_layers):
        for pos in range(n_pos):
            clean_act = clean_cache["resid_post", layer][0, pos].detach().clone()

            def patch_hook(value, hook, _clean=clean_act, _pos=pos):
                value[0, _pos] = _clean
                return value

            with torch.no_grad():
                patched_logits = model.run_with_hooks(
                    corr_toks,
                    fwd_hooks=[(f"blocks.{layer}.hook_resid_post", patch_hook)],
                )[0, -1]

            patched_diff = (patched_logits[correct_id] - patched_logits[foil_id]).item()
            denom = (clean_diff - baseline_diff) if abs(clean_diff - baseline_diff) > 1e-6 else 1.0
            patch_mat[layer, pos] = (patched_diff - baseline_diff) / denom

    model.reset_hooks()
    return patch_mat.cpu()


print(" Analysis 7: Activation Patching / Causal Tracing")
patch_rows = []
PATCH_CHECKPOINTS = ["chck_10M", "chck_50M", "chck_1000M"]

for rev in PATCH_CHECKPOINTS:
    print(f" {rev} ...")
    m = load_model(REPO_BASE, revision=rev)

    try:
        patch_mat = run_activation_patching(
            m,
            clean=PATCH_PROBE["clean"],
            corrupted=PATCH_PROBE["corrupted"],
            correct_w=PATCH_PROBE["correct_word"],
            foil_w=PATCH_PROBE["foil_word"],
        )

        torch.save(patch_mat, f"results_v2/data/analysis_7_patch_{rev}.pt")

        n_layers, n_pos = patch_mat.shape
        for layer in range(n_layers):
            for pos in range(n_pos):
                patch_rows.append({
                    "checkpoint": rev.replace("chck_", ""),
                    "layer": layer,
                    "position": pos,
                    "recovery": patch_mat[layer, pos].item(),
                })

        max_l, max_p = divmod(patch_mat.argmax().item(), n_pos)
        print(f" Causal hotspot: layer={max_l}, pos={max_p}, recovery={patch_mat.max():.3f}")

    except Exception as e:
        print(f" Patching failed at {rev}: {e}")

    release(m)
    print(f" VRAM after release: {vram_gb():.2f} GB")

df_patch = pd.DataFrame(patch_rows)
df_patch.to_csv("results_v2/data/analysis_7_activation_patching.csv", index=False)
print(" Analysis 7 complete - data saved.")

for ckpt in df_patch["checkpoint"].unique():
    sub = df_patch[df_patch["checkpoint"] == ckpt]
    pivot = sub.pivot(index="layer", columns="position", values="recovery")
    fig7 = px.imshow(
        pivot,
        color_continuous_scale="Blues",
        zmin=0, zmax=1,
        title=f"Analysis 7: Activation Patching Recovery @ {ckpt} words",
        labels={"x": "Token Position", "y": "Layer", "color": "Recovery (0=none, 1=full)"},
        template="plotly_white",
    )
    fig7.write_html(f"results_v2/figures/analysis_7_patch_{ckpt}.html")
    fig7.show()


## Summary Table
Human-readable digest of all seven analyses.

In [ ]:
commit_lookup_std = df_kl.groupby("checkpoint")["commit_standard"].first().to_dict()
commit_lookup_tun = df_kl.groupby("checkpoint")["commit_tuned"].first().to_dict()
peak_ind = df_induction.set_index("checkpoint")["max_score"].to_dict()
coref_mean_by_ckpt = df_coref.groupby("checkpoint")["logit_diff"].mean().to_dict()
entropy_median = df_entropy.groupby("checkpoint")["entropy"].median().to_dict()

summary_rows = []
for rev in ["10M", "20M", "50M", "200M", "1000M"]:
 summary_rows.append({
 "Checkpoint": f"{rev} words",
 "Peak Induction Score (v2)": f"{peak_ind.get(rev, ' - '):.3f}"
 if isinstance(peak_ind.get(rev), float) else " - ",
 "KL Commit Layer (std)": commit_lookup_std.get(rev, " - "),
 "KL Commit Layer (tuned)": commit_lookup_tun.get(rev, " - "),
 "Coref Logit Diff (mean)": f"{coref_mean_by_ckpt.get(rev, ' - '):.3f}"
 if isinstance(coref_mean_by_ckpt.get(rev), float) else " - ",
 "Attn Entropy (median)": f"{entropy_median.get(rev, ' - '):.3f}"
 if isinstance(entropy_median.get(rev), float) else " - ",
 })

for base_label in ["50M", "200M"]:
 sub = df_ppo[df_ppo["base"] == base_label]
 top_delta = sub["delta_induction"].abs().max()
 n_pos = (sub["delta_induction"] > 0).sum()
 n_neg = (sub["delta_induction"] < 0).sum()
 summary_rows.append({
 "Checkpoint": f"{base_label}+PPO (delta)",
 "Peak Induction Score (v2)": f"Δmax={top_delta:.3f} ↑{n_pos} ↓{n_neg} heads",
 "KL Commit Layer (std)": "n/a",
 "KL Commit Layer (tuned)": "n/a",
 "Coref Logit Diff (mean)": "n/a",
 "Attn Entropy (median)": "n/a",
 })

df_summary = pd.DataFrame(summary_rows)
df_summary.to_csv("results_v2/data/summary_table_v2.csv", index=False)
print(df_summary.to_string(index=False))
print(" Summary table saved.")


In [ ]:
shutil.make_archive("Kind_LM_Project_B_Results_v2", "zip", "results_v2")

try:
    from google.colab import files
    files.download("Kind_LM_Project_B_Results_v2.zip")
except ImportError:
    pass
